# Design Decisions (D1-D8)

These are my own answers to the "Questions to Think About" from the project brief.

### D1 - What happens if a new file arrives while the pipeline is running?
My pipeline uses `trigger(availableNow=True)`. This means: when a run starts, it looks at what files exist RIGHT NOW, processes only those, then stops. If a new file shows up while it's running, it is simply not included in that run. It just waits. The next time I run the notebook, the checkpoint sees that file as "new" and picks it up then. No data is lost, and nothing gets processed twice.

### D2 - Where should payment type and vendor ID be decoded (turned into readable words)?
In Silver. Bronze must stay exactly as it arrived (raw), so decoding there is not allowed. Gold would mean copying the same decode logic into 3 different tables, which is messy and easy to get wrong. Silver is the one correct place: decode once, and every table after it automatically gets the clean labels.

### D3 - Where is "which files were already processed" remembered? What if the compute restarts?
It's remembered in the checkpoint folder, which I store in a Volume (a storage location), not in the compute's memory. This is why restarting compute does not cause duplicate data - the checkpoint is just files sitting safely in storage. If I deleted the checkpoint folder on purpose, Spark would forget everything and reprocess all files from the start.

### D4 - How do I fix a Gold table if someone accidentally breaks it with a bad query?
I use Delta Lake's time travel feature. Every table write creates a new saved version. I can run `DESCRIBE HISTORY table_name` to see all past versions, then `RESTORE TABLE table_name TO VERSION AS OF <number>` to instantly go back to a good version. I don't need to rebuild anything from scratch.

### D5 - What do I do about bad rows, like 0 miles but a $50 fare, or dropoff before pickup?
I remove them using clear rules in Silver, and each rule has a comment explaining why (for example: "dropoff must happen after pickup, because a trip can't end before it starts"). Someone reading my code later doesn't have to guess - the reason is written right next to the rule.

### D6 - Why keep the raw (Bronze) data at all, once I have a cleaned version?
Because my cleaning rules might be wrong, or might need to change later. If I ever discover a mistake in my Silver rules, I need the original untouched data to fix it and rebuild Silver correctly. Without Bronze, I would have no safe original copy to go back to.

### D7 - What are the two trigger options, and which one fits this project?
`processingTime="..."` keeps a stream running forever, checking for new files every few seconds or minutes. `availableNow=True` processes whatever is available right now, then stops by itself. I used `availableNow` because it is the only one that works on Databricks Free Edition's serverless compute, and it is also cheaper and easier to control for a learning project.

### D8 - How do I add a new column to Silver later without deleting and rebuilding the whole table?
I add `.option("mergeSchema", "true")` to the write. This lets Delta safely add the new column to the table. All of the old rows keep their data, and they just get an empty (null) value for the new column.